# Lesson 9: Agent with Tools

## Why Do Agents Need Tools?

In the last lesson, our agent could only **think** and **respond** from its built-in knowledge.

But what if you ask: *"What are the latest SEO trends in 2026?"* — The agent doesn't know because its knowledge has a cutoff.

From Lesson 5: LLMs are trained on data up to a certain date — after that, they're blind. **Tools** solve this. They let an agent **do** more:
- Search the web
- Query databases
- Find images
- And much more...

The same way an employee can read the news and search Google, an agent with tools can search for real-time information.

> **Cost:** ~$0.02-0.10 per cell (Sonnet calls with web search). Run each cell once to keep costs low.
>
> **Requires:** `DATA_FOR_SEO_API_KEY` set in your `.env` file. If not set, search calls will fail but you can still read through the code.

**Setup cell below:** The first code cell adds our project's `output/backend` folder to Python's search path. This lets us `import` our project's agents and tools. You'll see this pattern in every lesson from now on — just run it and move on.

In [ ]:
# This line tells Python where our project files live (../../output/backend).
# You'll see it at the start of every lesson from now on — just run it.
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

from dotenv import load_dotenv
load_dotenv()

## DataForSEO — A Professional Search Tool

**DataForSEO** is the web search API used in our product. It queries Google and returns structured results.

Unlike free tools (which get rate-limited at scale), DataForSEO is a **paid API** used by professional SEO tools worldwide.

Our product wraps it in a **Toolkit** — a class that the agent can call:

```python
from tools.search import DataForSEOSearchTools
from tools.aio import get_dataforseo_credentials

creds = get_dataforseo_credentials()  # Reads DATA_FOR_SEO_API_KEY from .env
search = DataForSEOSearchTools(login=creds[0], password=creds[1])

agent = Agent(
    tools=[search],  # Give the agent search capability
)
```

`get_dataforseo_credentials()` decodes the API key from `.env` into a login/password pair. The `DataForSEOSearchTools` toolkit gives the agent a `web_search()` method it can call.

In [ ]:
from agno.agent import Agent
from agno.models.anthropic import Claude
from tools.search import DataForSEOSearchTools
from tools.aio import get_dataforseo_credentials

# Set up search tool with credentials from .env
creds = get_dataforseo_credentials()
if creds is None:
    print("WARNING: DATA_FOR_SEO_API_KEY not set. Search examples won't work.")
    print("You can still read through the code to understand the patterns.")
    search_tools = []
else:
    search_tools = [DataForSEOSearchTools(login=creds[0], password=creds[1])]
    print("DataForSEO configured! Search tools ready.")

# Agent with web search capability!
agent = Agent(
    name="Researcher",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=search_tools,
    instructions=["Search the web for information and summarize the results."],
)

response = agent.run("Find the top 3 SEO trends in 2026")
print(response.content)

## What Happened Behind the Scenes

When you ran the code above:

1. **You sent a question** — "Find the top 3 SEO trends in 2026"
2. **Agent decided** — "I need fresh info, let me use web_search"
3. **Agent called the tool** — Searched Google via DataForSEO API
4. **Agent read the results** — Processed the search results
5. **Agent summarized** — Wrote a response based on its findings

**Key point:** You don't write search code yourself. The agent **decides on its own** when to use its tools.

### Under the Hood: How Tool Calls Work

<details>
<summary>Click to expand — optional deeper understanding</summary>

When the agent "calls a tool," what actually happens is an **API call** — a request sent over the internet. Here's the simplified version:

**What is an API?** Think of a restaurant:
- You (the agent) place an **order** (request) with the waiter (API)
- The kitchen (server) prepares the food (data)
- The waiter brings it back to you (response)

DataForSEO is the kitchen. The agent sends a request ("search for 'SEO trends 2026'"), and DataForSEO sends back structured results.

**Two types of requests:**

| Type | When | Example |
|------|------|---------|
| **GET** | Fetching data | "Give me search results for this keyword" |
| **POST** | Sending data | "Here's my search query, process it and return results" |

**Status codes** tell you what happened:

| Code | Meaning | What to do |
|------|---------|-----------|
| 200 | Success | Process the results |
| 401 | Bad API key | Check your `.env` file |
| 404 | Not found | Check the endpoint URL |
| 429 | Too many requests | Wait and retry |
| 500 | Server error | Try again later |

**The response comes back as JSON** — structured data that Python can read as a dictionary:
```python
{"status": 200, "results": [{"title": "SEO Guide", "url": "..."}]}
```

You don't need to write any of this yourself — the `DataForSEOSearchTools` toolkit handles it all. But understanding what's happening helps you debug when things go wrong (e.g., "401 error? Check your API key").

</details>

In [ ]:
# Another example — research a specific topic
response = agent.run("What is SEONGON? Find information about this company.")
print(response.content)

## Comparing: Agent WITHOUT Tools vs WITH Tools

The difference:

| | Without Tools | With Tools |
|---|---|---|
| Knowledge | Only what it was trained on | Can search for new information |
| Information | May be outdated/wrong | Current and accurate |
| Capability | Think only | Think + Act |

Run the code below to see it in practice.

In [ ]:
# Agent WITHOUT tools
agent_no_tools = Agent(
    name="Agent Without Tools",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    instructions=["Answer concisely."],
)

# Agent WITH tools (same search_tools from above)
agent_with_tools = Agent(
    name="Agent With Tools",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=search_tools,
    instructions=["Search the web then answer concisely."],
)

question = "Which website currently ranks #1 for the keyword 'learn SEO' on Google?"

print("=== AGENT WITHOUT TOOLS ===")
r1 = agent_no_tools.run(question)
print(r1.content)

print("\n=== AGENT WITH TOOLS ===")
r2 = agent_with_tools.run(question)
print(r2.content)

## How the Product Uses Search

In the real product, the **Content Writer** agent uses DataForSEO search to research topics before writing articles.

From `output/backend/agents/content_writer.py`:
```python
from tools.search import DataForSEOSearchTools
from tools.aio import get_dataforseo_credentials

_tools = [save_article, list_all_articles]
_creds = get_dataforseo_credentials()
if _creds:
    _tools.insert(0, DataForSEOSearchTools(login=_creds[0], password=_creds[1]))
```

Notice the pattern: search is **conditional**. If `DATA_FOR_SEO_API_KEY` is not set, the Content Writer still works — it just writes from its built-in knowledge instead of searching first. The product degrades gracefully.

## Lesson 9 Summary

What you learned:
- **Tools** let agents **act**, not just think
- **DataForSEOSearchTools** — professional web search via the DataForSEO API
- Tools need **credentials** — `get_dataforseo_credentials()` reads from `.env`
- The agent **decides on its own** when to use its tools
- Agents with tools give **more accurate and up-to-date** results
- Tools can be **conditional** — the product works with or without search

**Next lesson:** We make agents return **structured data** (structured output) — instead of free-form text.

## Exercise

Create an agent with DataForSEO search tools and instructions that make it research **your company** or a competitor. Ask it a specific question like:
- "What services does [company] offer?"
- "Find recent news about [company]"

Print the response and check if the information is accurate.

In [ ]:
# Exercise: Fill in the blanks to create a research agent

from agno.agent import Agent
from agno.models.anthropic import Claude

# Create an agent that can search the web
company_researcher = Agent(
    name=___,                                    # Give it a name
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=___,                                   # Use search_tools from above
    instructions=[___],                          # Tell it to research and summarize
)

# Ask it to research a company
response = company_researcher.run(___)           # Ask about a company
print(response.content)

<details>
<summary>Click to reveal answer</summary>

```python
from agno.agent import Agent
from agno.models.anthropic import Claude

company_researcher = Agent(
    name="Company Researcher",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=search_tools,
    instructions=["Search the web for company information and summarize your findings clearly."],
)

response = company_researcher.run("What services does SEONGON offer? Find their website and key information.")
print(response.content)
```
</details>